# 03 Statistical Analysis

This notebook uses hypothesis testing to validate whether customer churn is related to pricing, contract type, and service category.

In [6]:
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
import statsmodels.stats.api as sms

In [7]:
df = pd.read_csv("../data/processed/cleaned_telco_churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnFlag,TenureGroup,MonthlyChargeGroup,RevenueSegment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0-12 Months,Low,Low Value
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,0,25-48 Months,Medium,Medium Value
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,0-12 Months,Medium,Low Value
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,25-48 Months,Medium,Medium Value
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,0-12 Months,High,Low Value


## Test 1: Independent t-test

Question: Do churned customers have significantly different monthly charges than non-churned customers?

H0: Average monthly charges are the same for churned and non-churned customers.  
H1: Average monthly charges are different for churned and non-churned customers.

In [8]:
churned = df[df["Churn"] == "Yes"]["MonthlyCharges"]
not_churned = df[df["Churn"] == "No"]["MonthlyCharges"]

t_stat, p_value = ttest_ind(churned, not_churned, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0: Monthly charges are significantly different.")
else:
    print("Fail to reject H0: No significant difference found.")

T-statistic: 18.407526676414673
P-value: 8.592449331547065e-73
Reject H0: Monthly charges are significantly different.


## Test 2: Chi-square Test

Question: Is contract type associated with churn?

H0: Contract type and churn are independent.  
H1: Contract type and churn are associated.

In [9]:
contingency_table = pd.crosstab(df["Contract"], df["Churn"])
contingency_table

Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


In [10]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)

if p_value < 0.05:
    print("Reject H0: Contract type is associated with churn.")
else:
    print("Fail to reject H0: No association found.")

Chi-square statistic: 1184.5965720837926
P-value: 5.863038300673391e-258
Degrees of freedom: 2
Reject H0: Contract type is associated with churn.


## Test 3: ANOVA

Question: Are monthly charges different across internet service types?

H0: Average monthly charges are the same across internet service types.  
H1: At least one internet service type has a different average monthly charge.

In [11]:
internet_groups = [
    group["MonthlyCharges"].values
    for name, group in df.groupby("InternetService")
]

f_stat, p_value = f_oneway(*internet_groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0: Monthly charges differ across internet service types.")
else:
    print("Fail to reject H0: No significant difference found.")

F-statistic: 16111.646283833992
P-value: 0.0
Reject H0: Monthly charges differ across internet service types.


## Confidence Interval

95% confidence interval for average monthly charges of churned customers.

In [12]:
confidence_interval = sms.DescrStatsW(churned).tconfint_mean()
print("95% Confidence Interval:", confidence_interval)

95% Confidence Interval: (np.float64(73.32234641761482), np.float64(75.5603181088699))


## Correlation

In [13]:
df[["tenure", "MonthlyCharges", "TotalCharges", "ChurnFlag"]].corr()

,tenure,MonthlyCharges,TotalCharges,ChurnFlag
tenure,1.000000,0.247900,0.825464,-0.352229
MonthlyCharges,0.247900,1.000000,0.650864,0.193356
TotalCharges,0.825464,0.650864,1.000000,-0.199037
ChurnFlag,-0.352229,0.193356,-0.199037,1.000000
